# Phase 0/2 - CUDA & Triton kernels (matmul, RMSNorm)

Compiles and **correctness-checks** the hand-written kernels on Colab's free T4: tiled `matmul` (CUDA), `rmsnorm` (CUDA), and fused RMSNorm (Triton). `vector_add` has its own notebook.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**, then Run all. Each kernel asserts against a PyTorch reference before reporting timing.

In [ ]:
!pip install -q ninja   # required by torch.utils.cpp_extension to compile .cu
!nvidia-smi --query-gpu=name --format=csv,noheader
import torch, triton
print("torch", torch.__version__, "| triton", triton.__version__)
assert torch.cuda.is_available(), "Set Runtime -> T4 GPU"

## 1. Tiled matmul (CUDA, shared-memory tiling)

In [ ]:
%%writefile matmul.cu
// matmul — Phase 0, kernel 2: tiled matrix multiply using shared memory.
//
// C = A @ B for A[M,K], B[K,N]. The point is SHARED-MEMORY TILING: each block
// loads a TILE x TILE tile of A and of B into __shared__ memory, __syncthreads(),
// accumulates the partial products, then advances to the next tile along K. Every
// A/B element is loaded from global memory once per tile and reused TILE times
// from fast shared memory — that reuse is the whole point. Compare vs cuBLAS
// (torch.matmul) in the bench.

#include <torch/extension.h>
#include <cuda_runtime.h>

#define TILE 16

__global__ void matmul_kernel(const float* A, const float* B, float* C,
                              int M, int N, int K) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;   // output row this thread computes
    int col = blockIdx.x * TILE + threadIdx.x;   // output col this thread computes
    float acc = 0.0f;

    for (int t = 0; t < (K + TILE - 1) / TILE; ++t) {
        int a_col = t * TILE + threadIdx.x;
        int b_row = t * TILE + threadIdx.y;
        // Cooperative load; zero-pad the edges so out-of-range tiles contribute 0.
        As[threadIdx.y][threadIdx.x] = (row < M && a_col < K) ? A[row * K + a_col] : 0.0f;
        Bs[threadIdx.y][threadIdx.x] = (b_row < K && col < N) ? B[b_row * N + col] : 0.0f;
        __syncthreads();

        for (int k = 0; k < TILE; ++k)
            acc += As[threadIdx.y][k] * Bs[k][threadIdx.x];
        __syncthreads();   // don't overwrite the tile until everyone's done with it
    }

    if (row < M && col < N)
        C[row * N + col] = acc;
}

torch::Tensor matmul(torch::Tensor a, torch::Tensor b) {
    TORCH_CHECK(a.is_cuda() && b.is_cuda(), "inputs must be CUDA tensors");
    TORCH_CHECK(a.dim() == 2 && b.dim() == 2, "inputs must be 2D");
    TORCH_CHECK(a.size(1) == b.size(0), "inner dims must match: A[M,K] @ B[K,N]");
    a = a.contiguous();
    b = b.contiguous();
    int M = a.size(0), K = a.size(1), N = b.size(1);
    auto c = torch::empty({M, N}, a.options());

    dim3 threads(TILE, TILE);
    dim3 blocks((N + TILE - 1) / TILE, (M + TILE - 1) / TILE);
    matmul_kernel<<<blocks, threads>>>(
        a.data_ptr<float>(), b.data_ptr<float>(), c.data_ptr<float>(), M, N, K);
    return c;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("matmul", &matmul, "tiled matmul A @ B (CUDA)");
}


In [ ]:
from torch.utils.cpp_extension import load
import torch, triton
mm = load(name="matmul_ext", sources=["matmul.cu"], extra_cuda_cflags=["-arch=sm_75"], verbose=False)

# Non-square, not a multiple of TILE=16 -> exercises the partial-tile boundary.
M, K, N = 200, 320, 168
a = torch.randn(M, K, device="cuda"); b = torch.randn(K, N, device="cuda")
out = mm.matmul(a, b); ref = a @ b
print("max abs diff:", (out - ref).abs().max().item())
assert out.shape == (M, N) and torch.allclose(out, ref, atol=1e-3, rtol=1e-3), "matmul WRONG"
print("matmul PASS (matches cuBLAS)")
t_mine = triton.testing.do_bench(lambda: mm.matmul(a, b))
t_cublas = triton.testing.do_bench(lambda: a @ b)
print(f"tiled kernel {t_mine:.3f} ms   |   cuBLAS {t_cublas:.3f} ms   (cuBLAS is the ceiling)")

## 2. RMSNorm (CUDA, one block per row + tree reduction)

In [ ]:
%%writefile rmsnorm.cu
// rmsnorm — Phase 0, kernel 4: RMSNorm (the normalization TinyLlama uses).
//
// For each row x: y = x / sqrt(mean(x^2) + eps) * weight. No mean-subtraction
// (unlike LayerNorm). One block per row: the threads cooperatively reduce the
// sum of squares in shared memory, then each thread scales its slice of the row.

#include <torch/extension.h>
#include <cuda_runtime.h>

__global__ void rmsnorm_kernel(const float* x, const float* w, float* y,
                               int N, float eps) {
    int row = blockIdx.x;
    int tid = threadIdx.x;
    int nthreads = blockDim.x;
    const float* xrow = x + (long)row * N;
    float* yrow = y + (long)row * N;

    // Each thread sums the squares of its strided slice of the row.
    extern __shared__ float sdata[];
    float local = 0.0f;
    for (int i = tid; i < N; i += nthreads) {
        float v = xrow[i];
        local += v * v;
    }
    sdata[tid] = local;
    __syncthreads();

    // Tree reduction to sdata[0] = sum of squares over the whole row.
    for (int s = nthreads / 2; s > 0; s >>= 1) {
        if (tid < s) sdata[tid] += sdata[tid + s];
        __syncthreads();
    }

    float inv = rsqrtf(sdata[0] / N + eps);   // 1 / sqrt(mean(x^2) + eps)
    for (int i = tid; i < N; i += nthreads)
        yrow[i] = xrow[i] * inv * w[i];
}

torch::Tensor rmsnorm(torch::Tensor x, torch::Tensor weight, double eps) {
    TORCH_CHECK(x.is_cuda() && weight.is_cuda(), "inputs must be CUDA tensors");
    TORCH_CHECK(x.dim() == 2, "x must be 2D [rows, N]");
    TORCH_CHECK(weight.numel() == x.size(1), "weight must match row length");
    x = x.contiguous();
    weight = weight.contiguous();
    int rows = x.size(0), N = x.size(1);
    auto y = torch::empty_like(x);

    const int threads = 256;   // power of two for the tree reduction
    rmsnorm_kernel<<<rows, threads, threads * sizeof(float)>>>(
        x.data_ptr<float>(), weight.data_ptr<float>(), y.data_ptr<float>(),
        N, (float)eps);
    return y;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("rmsnorm", &rmsnorm, "RMSNorm (CUDA)");
}


In [ ]:
from torch.utils.cpp_extension import load
import torch
rn = load(name="rmsnorm_ext", sources=["rmsnorm.cu"], extra_cuda_cflags=["-arch=sm_75"], verbose=False)

D, eps = 768, 1e-6   # GPT-2 hidden size
x = torch.randn(64, D, device="cuda"); w = torch.randn(D, device="cuda")
out = rn.rmsnorm(x, w, eps)
ref = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps) * w
print("max abs diff:", (out - ref).abs().max().item())
assert torch.allclose(out, ref, atol=1e-5), "rmsnorm WRONG"
print("rmsnorm (CUDA) PASS")

## 3. Fused RMSNorm (Triton, single fused kernel)

PyTorch RMSNorm is several ops (square, mean, rsqrt, mul, mul), each round-tripping activations through HBM. The Triton kernel reads the row once, normalizes in registers, writes once.

In [ ]:
import torch, triton
import triton.language as tl

@triton.jit
def _rmsnorm_kernel(x_ptr, w_ptr, y_ptr, stride_row, N, eps, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)
    offs = tl.arange(0, BLOCK_SIZE)
    mask = offs < N
    x = tl.load(x_ptr + row * stride_row + offs, mask=mask, other=0.0).to(tl.float32)
    ms = tl.sum(x * x, axis=0) / N
    x_norm = x * tl.rsqrt(ms + eps)
    w = tl.load(w_ptr + offs, mask=mask, other=0.0).to(tl.float32)
    tl.store(y_ptr + row * stride_row + offs, (x_norm * w).to(y_ptr.dtype.element_ty), mask=mask)

def fused_rmsnorm(x, weight, eps=1e-6):
    x = x.contiguous(); n_rows, N = x.shape
    y = torch.empty_like(x)
    _rmsnorm_kernel[(n_rows,)](x, weight, y, x.stride(0), N, eps,
                               BLOCK_SIZE=triton.next_power_of_2(N))
    return y

D, eps = 768, 1e-6
x = torch.randn(64, D, device="cuda"); w = torch.randn(D, device="cuda")
out = fused_rmsnorm(x, w, eps)
ref = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps) * w
print("max abs diff:", (out - ref).abs().max().item())
assert torch.allclose(out, ref, atol=1e-4), "triton rmsnorm WRONG"
print("fused RMSNorm (Triton) PASS")
# fusion win: one kernel + one read/write vs PyTorch's several-op sequence
t_fused = triton.testing.do_bench(lambda: fused_rmsnorm(x, w, eps))
t_torch = triton.testing.do_bench(lambda: x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps) * w)
print(f"fused {t_fused*1e3:.1f} us   |   pytorch (unfused) {t_torch*1e3:.1f} us")

## Done

If every cell printed PASS, all Bullet-1 kernels are verified correct on GPU. Paste the output back and I'll note the numbers in BENCHMARKS.md.